# OPIXray-v2 X-Ray Prohibited Item Detection: Unified Benchmark and DOAM Study

**COMP9444 Group Project**

This is a team project. The original notebook attributes the YOLO26 pipeline to the portfolio owner; the remaining detector pipelines, DOAM design, and unified evaluation were collaborative work. This portfolio copy removes teammates’ names and student IDs.

This notebook uses the OPIXray-v2 dataset to train and compare three different detectors(Faster R-CNN, YOLO26, RT-DETR) under the same data split and evaluation measures. Based on this, a De-occlusion Attention Module (DOAM) is added to each, so every detector is measured with and without this module under the same setting.


### Research questions
- RQ1. Do modern detectors (YOLO26, RT-DETR) outperform the Faster R-CNN baseline on OPIXray-v2?
- RQ2. Can DOAM be transferred to modern CNN and Transformer detectors?
- RQ3. Does DOAM help more on heavily occluded samples (OL3 vs OL1)?
- RQ4. Is DOAM's accuracy change worth its extra parameters, compute and latency?


### Reproducibility notes
- A single global `SEED` is set everywhere.
- The split is fixed by `train_ids.txt`, `val_ids.txt`, `test_ids.txt`; both the COCO and YOLO formats are derived from it, so no model re-splits the data.
- Every model is scored by the same `pycocotools` evaluator on the same test set.
- Training is guarded by `RUN_TRAINING`; set a variant to `False` to load its checkpoint instead of retraining.

---
## Section 0: Data Import and Global Configuration

This section defines the paths, experiment settings, dataset split, and data formats used by all models.

In [ ]:
import os, sys, json, time, random, platform, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

warnings.filterwarnings("ignore")

# Paths
ROOT          = Path.cwd().resolve()
OPIXRAY_RAW   = ROOT / "OPIXray"
COCO_DIR      = ROOT / "coco_format"
YOLO_DIR      = ROOT / "yolo_dataset"
IDS_DIR       = ROOT

OUTPUT_ROOT   = ROOT / "outputs"
CKPT_DIR      = OUTPUT_ROOT / "checkpoints"
PRED_DIR      = OUTPUT_ROOT / "predictions"
FIG_DIR       = OUTPUT_ROOT / "figures"
for d in (OUTPUT_ROOT, CKPT_DIR, PRED_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

COCO_ANN = {
    "train": COCO_DIR / "annotations" / "instances_train.json",
    "val":   COCO_DIR / "annotations" / "instances_val.json",
    "test":  COCO_DIR / "annotations" / "instances_test.json",
}
COCO_IMG = {"train": COCO_DIR / "train2017", "val": COCO_DIR / "val2017", "test": COCO_DIR / "test2017"}

CLASS_NAMES = ["Folding_Knife", "Multi-tool_Knife", "Scissor", "Straight_Knife", "Utility_Knife"]
NUM_CLASSES = len(CLASS_NAMES)

# Global experiment configuration
SEED       = 42
IMGSZ      = 640
EPOCHS     = 30
CONF_THRES = 0.001
IOU_NMS    = 0.6
MAX_DET    = 100

# Per-model batch size
BATCH = {"faster_rcnn": 8, "yolo26": 16, "rtdetr": 8}
NUM_WORKERS = 0
CACHE_IMAGES = True
FRCNN_PATIENCE = 5
FRCNN_MIN_EPOCHS = 15
FRCNN_MIN_DELTA = 0.001

# Run switches
RUN_TRAINING = {
    "faster_rcnn_baseline": False, "faster_rcnn_doam": False,
    "yolo26_baseline": False,      "yolo26_doam": False, "yolo26_model_specific": False,
    "rtdetr_baseline": False,      "rtdetr_doam": False, "rtdetr_model_specific": False,
}

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

print("Project root :", ROOT)
print("Device       :", DEVICE, "| torch", torch.__version__)
print("Classes      :", CLASS_NAMES)
print("Seed / imgsz / epochs:", SEED, IMGSZ, EPOCHS)

### Data import and split

The following cell reads the original OPIXray files, creates the training, validation, and test split, and prepares the COCO and YOLO data formats. If the split and formatted data already exist, it checks and uses them.

In [ ]:
# Set False only when deliberately managing all split/format files outside this notebook.
AUTO_PREPARE_DATA = True

import shutil
import tempfile

_SPLITS = ("train", "val", "test")
_ID_PATHS = {s: IDS_DIR / f"{s}_ids.txt" for s in _SPLITS}
_RAW = {
    "train_images": OPIXRAY_RAW / "train" / "train_image",
    "train_annotations": OPIXRAY_RAW / "train" / "train_annotation",
    "test_images": OPIXRAY_RAW / "test" / "test_image",
    "test_annotations": OPIXRAY_RAW / "test" / "test_annotation",
    "official_train": OPIXRAY_RAW / "train" / "train_knife.txt",
    "official_test": OPIXRAY_RAW / "test" / "test_knife.txt",
}

def _nonempty_lines(path):
    return [line.strip() for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

def _load_or_create_splits():
    present = {s: _ID_PATHS[s].is_file() for s in _SPLITS}
    if any(present.values()) and not all(present.values()):
        missing = [p.name for s, p in _ID_PATHS.items() if not present[s]]
        raise RuntimeError(f"Partial canonical split found; refusing to mix splits. Missing: {missing}")

    if all(present.values()):
        ids = {s: _nonempty_lines(_ID_PATHS[s]) for s in _SPLITS}
        action = "reused existing canonical split"
    else:
        for key in ("official_train", "official_test"):
            if not _RAW[key].is_file():
                raise FileNotFoundError(f"Required official OPIXray list not found: {_RAW[key]}")
        official_train = _nonempty_lines(_RAW["official_train"])
        official_test = _nonempty_lines(_RAW["official_test"])
        shuffled = official_train.copy()
        random.Random(SEED).shuffle(shuffled)
        cut = int(len(shuffled) * 0.8)
        ids = {"train": shuffled[:cut], "val": shuffled[cut:], "test": official_test.copy()}
        for split in _SPLITS:
            tmp = _ID_PATHS[split].with_suffix(".txt.tmp")
            tmp.write_text("\n".join(ids[split]) + "\n", encoding="utf-8")
            tmp.replace(_ID_PATHS[split])
        action = "created deterministic canonical split"

    sets = {s: set(ids[s]) for s in _SPLITS}
    if len(sets["train"]) != len(ids["train"]) or len(sets["val"]) != len(ids["val"]) or len(sets["test"]) != len(ids["test"]):
        raise ValueError("Duplicate image IDs found in a canonical split")
    if sets["train"] & sets["val"] or sets["train"] & sets["test"] or sets["val"] & sets["test"]:
        raise ValueError("Canonical train/val/test splits overlap")
    if _RAW["official_train"].is_file() and _RAW["official_test"].is_file():
        official_train = _nonempty_lines(_RAW["official_train"])
        official_test = _nonempty_lines(_RAW["official_test"])
        if sets["train"] | sets["val"] != set(official_train):
            raise ValueError("train_ids + val_ids do not exactly reproduce the official OPIXray train set")
        if sets["test"] != set(official_test):
            raise ValueError("test_ids does not exactly reproduce the official frozen OPIXray test set")
    elif all(present.values()):
        print("[data] official raw split lists unavailable; validated existing split internally only")
    print(f"[data] {action}:", {s: len(ids[s]) for s in _SPLITS})
    return ids

def _source_dirs(split):
    prefix = "test" if split == "test" else "train"
    return _RAW[f"{prefix}_images"], _RAW[f"{prefix}_annotations"]

def _validate_raw(ids):
    missing = []
    for split in _SPLITS:
        img_dir, ann_dir = _source_dirs(split)
        if not img_dir.is_dir() or not ann_dir.is_dir():
            missing.append(f"{split}: {img_dir} or {ann_dir}")
            continue
        for image_id in ids[split]:
            if not (img_dir / f"{image_id}.jpg").is_file(): missing.append(str(img_dir / f"{image_id}.jpg"))
            if not (ann_dir / f"{image_id}.txt").is_file(): missing.append(str(ann_dir / f"{image_id}.txt"))
            if len(missing) >= 10: break
        if len(missing) >= 10: break
    if missing:
        raise FileNotFoundError("Raw OPIXray data is incomplete (first missing entries):\n" + "\n".join(missing))

def _parse_raw_annotation(path):
    boxes = []
    for line_no, line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip(): continue
        parts = line.split()
        if len(parts) < 6:
            raise ValueError(f"Malformed annotation {path}:{line_no}: {line!r}")
        cls = parts[1]
        if cls not in CLASS_NAMES:
            raise ValueError(f"Unknown class {cls!r} in {path}:{line_no}")
        x1, y1, x2, y2 = map(float, parts[2:6])
        if x2 <= x1 or y2 <= y1:
            raise ValueError(f"Non-positive box in {path}:{line_no}")
        boxes.append((cls, x1, y1, x2, y2))
    return boxes

def _link_or_copy(src, dst):
    try:
        os.link(src, dst)
        return "hardlink"
    except OSError:
        try:
            Path(dst).symlink_to(Path(src).resolve())
            return "symlink"
        except OSError:
            shutil.copy2(src, dst)
            return "copy"

def _image_stems(folder):
    if not Path(folder).is_dir(): return None
    return {p.stem for p in Path(folder).iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}

def _format_state(kind, ids):
    root = COCO_DIR if kind == "COCO" else YOLO_DIR
    if not root.exists(): return "absent"
    try:
        for split in _SPLITS:
            expected = set(ids[split])
            if kind == "COCO":
                img_dir = root / {"train": "train2017", "val": "val2017", "test": "test2017"}[split]
                ann_path = root / "annotations" / f"instances_{split}.json"
                if _image_stems(img_dir) != expected or not ann_path.is_file(): return "partial"
                payload = json.loads(ann_path.read_text(encoding="utf-8"))
                if {str(x["id"]).zfill(6) for x in payload.get("images", [])} != expected: return "partial"
            else:
                if _image_stems(root / "images" / split) != expected: return "partial"
                label_dir = root / "labels" / split
                if not label_dir.is_dir() or {p.stem for p in label_dir.glob("*.txt")} != expected: return "partial"
        return "valid"
    except (OSError, ValueError, KeyError, TypeError):
        return "partial"

def _build_yolo(tmp_root, ids):
    class_to_id = {name: i for i, name in enumerate(CLASS_NAMES)}
    modes = defaultdict(int)
    for split in _SPLITS:
        out_images, out_labels = tmp_root / "images" / split, tmp_root / "labels" / split
        out_images.mkdir(parents=True); out_labels.mkdir(parents=True)
        src_images, src_annotations = _source_dirs(split)
        for image_id in ids[split]:
            src_img = src_images / f"{image_id}.jpg"
            modes[_link_or_copy(src_img, out_images / src_img.name)] += 1
            with Image.open(src_img) as im: width, height = im.size
            rows = []
            for cls, x1, y1, x2, y2 in _parse_raw_annotation(src_annotations / f"{image_id}.txt"):
                cx, cy = (x1 + x2) / (2 * width), (y1 + y2) / (2 * height)
                bw, bh = (x2 - x1) / width, (y2 - y1) / height
                rows.append(f"{class_to_id[cls]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            (out_labels / f"{image_id}.txt").write_text("\n".join(rows) + ("\n" if rows else ""), encoding="utf-8")
    (tmp_root / "dataset.yaml").write_text(
        f"path: {YOLO_DIR.resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\n\n"
        f"nc: {NUM_CLASSES}\nnames: {CLASS_NAMES}\n", encoding="utf-8")
    return dict(modes)

def _build_coco(tmp_root, ids):
    class_to_id = {name: i for i, name in enumerate(CLASS_NAMES)}
    modes = defaultdict(int)
    (tmp_root / "annotations").mkdir(parents=True)
    split_dirs = {"train": "train2017", "val": "val2017", "test": "test2017"}
    for split in _SPLITS:
        out_images = tmp_root / split_dirs[split]; out_images.mkdir(parents=True)
        src_images, src_annotations = _source_dirs(split)
        payload = {"images": [], "annotations": [],
                   "categories": [{"id": i, "name": name} for i, name in enumerate(CLASS_NAMES)]}
        ann_id = 0
        for image_id in ids[split]:
            src_img = src_images / f"{image_id}.jpg"
            modes[_link_or_copy(src_img, out_images / src_img.name)] += 1
            with Image.open(src_img) as im: width, height = im.size
            numeric_id = int(image_id)
            payload["images"].append({"id": numeric_id, "file_name": src_img.name, "width": width, "height": height})
            for cls, x1, y1, x2, y2 in _parse_raw_annotation(src_annotations / f"{image_id}.txt"):
                bw, bh = x2 - x1, y2 - y1
                payload["annotations"].append({"id": ann_id, "image_id": numeric_id,
                    "category_id": class_to_id[cls], "bbox": [x1, y1, bw, bh],
                    "area": bw * bh, "iscrowd": 0})
                ann_id += 1
        out_json = tmp_root / "annotations" / f"instances_{split}.json"
        out_json.write_text(json.dumps(payload), encoding="utf-8")
    return dict(modes)

def _ensure_format(kind, ids):
    state = _format_state(kind, ids)
    if state == "valid":
        print(f"[data] reused existing valid {kind} dataset")
        return
    target = COCO_DIR if kind == "COCO" else YOLO_DIR
    if state == "partial":
        raise RuntimeError(f"{kind} dataset at {target} is partial or inconsistent. "
                           "It was left untouched; move/remove it deliberately, then rerun this cell.")
    tmp_root = Path(tempfile.mkdtemp(prefix=f".{target.name}_build_", dir=ROOT))
    try:
        modes = _build_coco(tmp_root, ids) if kind == "COCO" else _build_yolo(tmp_root, ids)
        tmp_root.replace(target)
        if _format_state(kind, ids) != "valid":
            raise RuntimeError(f"Generated {kind} dataset failed post-build validation")
        print(f"[data] created {kind} dataset ({modes})")
    except Exception:
        if tmp_root.exists(): shutil.rmtree(tmp_root)
        raise

def ensure_opixray_data_ready():
    if not AUTO_PREPARE_DATA:
        print("[data] automatic raw-data bootstrap disabled")
        return
    ids = _load_or_create_splits()
    states = {kind: _format_state(kind, ids) for kind in ("COCO", "YOLO")}
    if any(state == "absent" for state in states.values()):
        _validate_raw(ids)
    _ensure_format("COCO", ids)
    _ensure_format("YOLO", ids)
    print("[data] OPIXray split and detector formats are ready")

ensure_opixray_data_ready()

### Canonical split and leakage check

OPIXray-v2 has an official train (7,109) and test (1,776) split. We keep the official test set unchanged and take a validation set from the official training set:

- train = 5,687, val = 1,422 (about 80/20 of the 7,109), test = 1,776 (official, frozen).

The split is fixed by `train_ids.txt`, `val_ids.txt`, `test_ids.txt`. Below we check that (a) the three id sets are disjoint (no leakage) and (b) both the COCO and YOLO formats contain exactly those images, so every model sees the same data.

In [ ]:
def read_ids(name):
    return [ln.strip() for ln in (IDS_DIR / name).read_text().splitlines() if ln.strip()]

split_ids = {"train": read_ids("train_ids.txt"),
             "val":   read_ids("val_ids.txt"),
             "test":  read_ids("test_ids.txt")}

# leakage check
sets = {k: set(v) for k, v in split_ids.items()}
assert not (sets["train"] & sets["val"]),  "LEAK: train/val overlap"
assert not (sets["train"] & sets["test"]), "LEAK: train/test overlap"
assert not (sets["val"]   & sets["test"]), "LEAK: val/test overlap"

# format alignment check (counts must match the id lists)
def count_images(folder):
    return sorted(p.stem for p in Path(folder).iterdir()
                  if p.suffix.lower() in {".jpg", ".jpeg", ".png"})

summary = []
for split in ("train", "val", "test"):
    n_ids   = len(split_ids[split])
    n_coco  = len(count_images(COCO_IMG[split]))
    n_yolo  = len(count_images(YOLO_DIR / "images" / split))
    summary.append({"split": split, "n_ids": n_ids, "coco_images": n_coco, "yolo_images": n_yolo})

split_df = pd.DataFrame(summary)
print("No train/val/test leakage detected.")
display(split_df)
assert (split_df["n_ids"] == split_df["coco_images"]).all(), "COCO images do not match id lists"
assert (split_df["n_ids"] == split_df["yolo_images"]).all(), "YOLO images do not match id lists"
print("COCO and YOLO formats are both aligned to the same canonical split.")

---
## Section 1: Exploratory Data Analysis and Preprocessing

We analyse the properties that most affect a detector: class balance, bounding-box size and shape, number of objects per image, and image resolution. Each plot is followed by what it implies for training.

In [ ]:
from pycocotools.coco import COCO

def load_ann_table(split):
    coco = COCO(str(COCO_ANN[split]))
    rows = []
    id2name = {c["id"]: c["name"] for c in coco.loadCats(coco.getCatIds())}
    for img_id in coco.getImgIds():
        info = coco.loadImgs(img_id)[0]
        anns = coco.loadAnns(coco.getAnnIds(imgIds=img_id))
        for a in anns:
            x, y, w, h = a["bbox"]
            rows.append({
                "split": split, "image_id": img_id,
                "img_w": info["width"], "img_h": info["height"],
                "class": id2name[a["category_id"]],
                "bw": w, "bh": h, "area": w * h,
                "aspect": w / h if h > 0 else np.nan,
                "rel_area": (w * h) / (info["width"] * info["height"]),
            })
    n_obj = {i: len(coco.getAnnIds(imgIds=i)) for i in coco.getImgIds()}
    return pd.DataFrame(rows), n_obj

ann_frames, obj_counts = {}, {}
for s in ("train", "val", "test"):
    ann_frames[s], obj_counts[s] = load_ann_table(s)
ann_all = pd.concat(ann_frames.values(), ignore_index=True)
print("Total annotated objects:", len(ann_all))
print("Objects per split:", {s: len(df) for s, df in ann_frames.items()})

In [ ]:
# Class distribution per split
fig, ax = plt.subplots(figsize=(9, 4.5))
ct = (ann_all.groupby(["class", "split"]).size().unstack(fill_value=0)
      .reindex(CLASS_NAMES))
ct = ct[["train", "val", "test"]]
ct.plot(kind="bar", ax=ax)
ax.set_title("Class distribution (number of bounding boxes) per split")
ax.set_xlabel("Class"); ax.set_ylabel("Number of boxes")
plt.xticks(rotation=20, ha="right"); plt.tight_layout()
plt.savefig(FIG_DIR / "eda_class_distribution.png", dpi=150, bbox_inches="tight"); plt.show()
display(ct)

In [ ]:
# Objects per image
opi = pd.Series([v for s in ("train","val","test") for v in obj_counts[s].values()])
fig, ax = plt.subplots(figsize=(6, 4))
opi.value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_title("Objects per image (all splits)")
ax.set_xlabel("Number of objects in an image"); ax.set_ylabel("Number of images")
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_objects_per_image.png", dpi=150, bbox_inches="tight"); plt.show()
print("Mean objects/image: %.2f | max: %d" % (opi.mean(), opi.max()))

In [ ]:
# Relative object size
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(np.sqrt(ann_all["rel_area"]), bins=40)
ax.set_title("sqrt(box area / image area)")
ax.set_xlabel("relative size"); ax.set_ylabel("count")
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_bbox_size.png", dpi=150, bbox_inches="tight"); plt.show()

# Bbox area per class (log)
fig, ax = plt.subplots(figsize=(8, 4))
data = [ann_all.loc[ann_all["class"] == c, "area"].values for c in CLASS_NAMES]
ax.boxplot(data, tick_labels=CLASS_NAMES, showfliers=False)
ax.set_yscale("log"); ax.set_title("Bounding-box area per class (log scale)")
ax.set_ylabel("area (px^2)"); plt.xticks(rotation=20, ha="right")
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_area_per_class.png", dpi=150, bbox_inches="tight"); plt.show()

print("Median relative box size: %.4f  (small relative to 1225x954 images)" % ann_all["rel_area"].median())

In [ ]:
tr = ann_frames["train"]
fig, ax = plt.subplots(figsize=(7, 6))
for c in CLASS_NAMES:
    sub = tr[tr["class"] == c]
    ax.scatter(sub["bw"], sub["bh"], s=8, alpha=0.4, label=c)
ax.set_xlabel("box width (px)"); ax.set_ylabel("box height (px)")
ax.set_title("Bounding-box width vs height by class (train)")
ax.legend(markerscale=2, fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "eda_bbox_scatter.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Ground-truth visualisation grid
def show_gt_grid(split="train", n=6):
    coco = COCO(str(COCO_ANN[split]))
    id2name = {c["id"]: c["name"] for c in coco.loadCats(coco.getCatIds())}
    ids = coco.getImgIds()
    rng = random.Random(SEED); chosen = rng.sample(ids, min(n, len(ids)))
    cols = 3; rows = int(np.ceil(len(chosen)/cols))
    plt.figure(figsize=(15, 5*rows))
    for i, img_id in enumerate(chosen, 1):
        info = coco.loadImgs(img_id)[0]
        img = Image.open(COCO_IMG[split] / info["file_name"]).convert("RGB")
        draw = ImageDraw.Draw(img)
        for a in coco.loadAnns(coco.getAnnIds(imgIds=img_id)):
            x, y, w, h = a["bbox"]
            draw.rectangle([x, y, x+w, y+h], outline=(255, 0, 0), width=4)
            draw.text((x, max(0, y-12)), id2name[a["category_id"]], fill=(255, 0, 0))
        ax = plt.subplot(rows, cols, i); ax.imshow(img); ax.axis("off"); ax.set_title(info["file_name"])
    plt.tight_layout(); plt.savefig(FIG_DIR / "eda_gt_examples.png", dpi=120, bbox_inches="tight"); plt.show()

show_gt_grid("train", 6)

**EDA takeaways for training:**
- Class balance: the five classes are close to balanced except Straight_Knife, which has about half as many instances as the others (11.7% of boxes). It is reported separately in the per-class results.
- Small objects: the median box covers under 1% of the ~1225x954 image, so input resolution matters and aggressive down-scaling reduces recall.
- About one object per image: OPIXray images are mostly single-instance, so each miss costs recall directly.
- Appearance similarity: the five cutter classes look alike under X-ray, which produces class confusion (see the confusion matrix and per-class AP in Section 6).

In [ ]:
# Preprocessing sanity checks
def check_coco(split):
    coco = COCO(str(COCO_ANN[split])); issues = 0
    for img_id in coco.getImgIds():
        info = coco.loadImgs(img_id)[0]
        for a in coco.loadAnns(coco.getAnnIds(imgIds=img_id)):
            x, y, w, h = a["bbox"]
            if x < 0 or y < 0 or x+w > info["width"]+1 or y+h > info["height"]+1: issues += 1
            if not (0 <= a["category_id"] < NUM_CLASSES): issues += 1
            if w <= 0 or h <= 0: issues += 1
    return issues

def check_yolo(split):
    lbl_dir = YOLO_DIR / "labels" / split; issues = 0
    for p in lbl_dir.glob("*.txt"):
        for line in p.read_text().splitlines():
            if not line.strip(): continue
            c, cx, cy, bw, bh = map(float, line.split())
            if not (0 <= int(c) < NUM_CLASSES): issues += 1
            if not all(0.0 <= v <= 1.0 for v in (cx, cy, bw, bh)): issues += 1
    return issues

for s in ("train", "val", "test"):
    print(f"{s:5s}  COCO issues: {check_coco(s):3d}   YOLO issues: {check_yolo(s):3d}")
print("(0 issues everywhere means bbox coords are in-bounds, class ids valid, YOLO labels normalised.)")

---
## Section 2: DOAM and Model Evaluation

### 2.1 DOAM implementation

DOAM processes each image before it enters the detector. The edge branch applies Sobel filters to the blue channel. The material branch combines the RGB image with the edge map. Material features are pooled at three scales and combined with gated convolution to produce an attention map.

This implementation accepts a three channel image and returns a three channel image with the same height and width:

```
image (3xHxW) -> [ DOAM ] -> enhanced image (3xHxW) -> detector backbone
```

The baseline uses the original image. The DOAM variant uses the processed image. DOAM is part of the model and is updated during training.

In [ ]:
import torch.nn as nn
import torch.nn.functional as Fnn

class GatedConv2d(nn.Module):
    """Convolution multiplied by a learned sigmoid gate."""
    def __init__(self, in_ch, out_ch, k=3, padding=1, batch_norm=True):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, k, padding=padding)
        self.gate = nn.Conv2d(in_ch, out_ch, k, padding=padding)
        self.bn = nn.BatchNorm2d(out_ch) if batch_norm else None
    def forward(self, x):
        out = self.conv(x) * torch.sigmoid(self.gate(x))
        return self.bn(out) if self.bn is not None else out

class DOAM(nn.Module):
    """Three channel DOAM image module used before each detector."""
    def __init__(self, gamma_init: float = 0.1):
        super().__init__()
        kx = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
        ky = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
        self.register_buffer("sobel_x", kx)
        self.register_buffer("sobel_y", ky)
        # Edge branch: 1 -> 8 -> 16 -> 16 -> 8 -> 1
        self.edge_cnn = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 8, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(8, 1, 3, padding=1),
        )
        # Material branch: RGB and edge -> 8 -> 16 -> 32 -> 16 -> 8
        self.mat_cnn = nn.Sequential(
            nn.Conv2d(4, 8, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, 8, 3, padding=1), nn.ReLU(inplace=True),
        )
        # Three pooled scales followed by gated fusion
        self.ria_reduce = nn.ModuleList([nn.Conv2d(16, 8, 3, padding=1) for _ in range(3)])
        self.ria_gate = GatedConv2d(24, 8, 3, padding=1, batch_norm=True)
        # Material and edge features produce one attention map
        self.attn_head = nn.Conv2d(9, 1, 3, padding=1)
        nn.init.constant_(self.attn_head.bias, 2.0)
        self.gamma = nn.Parameter(torch.tensor(float(gamma_init)))

    def _sobel_mag(self, x1):
        gx = Fnn.conv2d(x1, self.sobel_x, padding=1)
        gy = Fnn.conv2d(x1, self.sobel_y, padding=1)
        return torch.sqrt(gx * gx + gy * gy + 1e-6)

    def _ria(self, feat):
        h, w = feat.shape[-2:]
        outs = []
        for i, k in enumerate((5, 10, 15)):
            ph, pw = (h + k - 1) // k, (w + k - 1) // k
            pooled = Fnn.avg_pool2d(feat, kernel_size=k, stride=k, ceil_mode=True, count_include_pad=False)
            n, c = pooled.shape[:2]
            up = pooled[:, :, :, None, :, None].expand(n, c, ph, k, pw, k).reshape(n, c, ph * k, pw * k)
            outs.append(self.ria_reduce[i](torch.cat([feat, up[:, :, :h, :w]], dim=1)))
        return self.ria_gate(torch.cat(outs, dim=1))

    def _attention(self, im):
        blue = im[:, 2:3]
        edge = self._sobel_mag(blue)
        edge_feat = self.edge_cnn(edge)
        emin = edge.amin(dim=(2, 3), keepdim=True)
        emax = edge.amax(dim=(2, 3), keepdim=True)
        edge_norm = (edge - emin) / (emax - emin + 1e-6)
        mat = self.mat_cnn(torch.cat([im, edge_norm], dim=1))
        mat = self._ria(mat)
        fused = self.attn_head(torch.cat([mat, edge_feat], dim=1))
        return torch.sigmoid(fused)

    def forward(self, im):
        attn = self._attention(im)
        return self.gamma * (attn * im) + (1.0 - self.gamma) * im

    @torch.no_grad()
    def attention_map(self, im):
        return self._attention(im)

def build_front(kind: str) -> nn.Module:
    return nn.Identity() if kind == "baseline" else DOAM()


In [ ]:
# Check output shape, gradients, and parameter count.
_doam = DOAM(gamma_init=0.5).to(DEVICE)
_x = torch.randn(1, 3, 64, 64, device=DEVICE, requires_grad=True)
_y = _doam(_x)
assert _y.shape == _x.shape, "DOAM must preserve shape"
_y.sum().backward()
n_params = sum(p.numel() for p in _doam.parameters() if p.requires_grad)
print("DOAM output shape :", tuple(_y.shape), "(== input, OK)")
print("Input grad flows  :", _x.grad is not None)
print("DOAM trainable params:", f"{n_params:,}")
del _doam, _x, _y

### 2.2 Model integration

Faster R-CNN uses a wrapper that applies either `nn.Identity` or `DOAM` before the detector. YOLO26 and RT-DETR add DOAM as the first layer of the model configuration. Their layer references and pretrained weight keys are adjusted for the added layer.

Within each detector, the baseline and DOAM variants use the same data and training settings.

In [ ]:
class DetectorWithFrontEnd(nn.Module):
    """Wrap a torchvision detection model with a shape-preserving front module."""
    def __init__(self, detector, front: nn.Module):
        super().__init__()
        self.front = front
        self.detector = detector
    def forward(self, images, targets=None):
        images = [self.front(img.unsqueeze(0)).squeeze(0) for img in images]
        return self.detector(images, targets) if targets is not None else self.detector(images)

### 2.3 Test evaluation

Each detector writes its test predictions to COCO JSON with `image_id`, `category_id`, `bbox`, and `score`. All test prediction files are evaluated with `pycocotools.COCOeval` against `instances_test.json`.

`RESULTS` stores one row for each model and variant. Section 6 uses these rows for the results table. The training curves in Section 6.1 use the validation records produced during training and are used only to show convergence.

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import io, contextlib

RESULTS = []
PRED_STORE = {}

def coco_eval(pred_json_path, gt_json_path=COCO_ANN["test"], img_ids=None):
    """Score a COCO prediction file with pycocotools. Returns a standardized metrics dict."""
    preds = json.loads(Path(pred_json_path).read_text())
    if len(preds) == 0:
        return {"mAP50": 0.0, "mAP50_95": 0.0, "recall": 0.0,
                "per_class_AP50": {c: 0.0 for c in CLASS_NAMES}}
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(str(gt_json_path))
        dt = gt.loadRes(preds)
        ev = COCOeval(gt, dt, iouType="bbox")
        if img_ids is not None:
            ev.params.imgIds = list(img_ids)
        ev.evaluate(); ev.accumulate(); ev.summarize()
    iou50 = 0
    prec = ev.eval["precision"]
    per_class = {}
    for k, c in enumerate(CLASS_NAMES):
        p = prec[iou50, :, k, 0, -1]
        per_class[c] = float(p[p > -1].mean()) if (p > -1).any() else 0.0
    rec = ev.eval["recall"][:, :, 0, -1]
    return {
        "mAP50":    float(ev.stats[1]),
        "mAP50_95": float(ev.stats[0]),
        "recall":   float(rec[rec > -1].mean()) if (rec > -1).any() else 0.0,
        "per_class_AP50": per_class,
    }

def measure_latency(predict_one, sample_images, warmup=5, n=50):
    """predict_one: fn(path)->None. Returns dict(ms_per_image, fps)."""
    imgs = list(sample_images[: warmup + n])
    for p in imgs[:warmup]:
        predict_one(p)
    measured = imgs[warmup:]
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for p in measured:
        predict_one(p)
    if DEVICE == "cuda": torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / max(1, len(measured))
    return {"latency_ms": dt * 1000, "fps": 1.0 / dt if dt > 0 else float("nan")}

def register_result(model, variant, pred_json, n_params=None, latency=None):
    """Evaluate a prediction json and append a standardized row to RESULTS."""
    m = coco_eval(pred_json)
    row = {"model": model, "variant": variant,
           "mAP50": m["mAP50"], "mAP50_95": m["mAP50_95"], "recall": m["recall"],
           "params_M": (n_params/1e6) if n_params else None,
           "latency_ms": latency["latency_ms"] if latency else None,
           "fps": latency["fps"] if latency else None}
    for c in CLASS_NAMES:
        row[f"AP50_{c}"] = m["per_class_AP50"][c]
    RESULTS[:] = [r for r in RESULTS if not (r["model"] == model and r["variant"] == variant)]
    RESULTS.append(row)
    PRED_STORE[(model, variant)] = str(pred_json)
    print(f"[registered] {model} / {variant}:  mAP50={m['mAP50']:.4f}  mAP50-95={m['mAP50_95']:.4f}")
    return row

def reuse_saved_prediction(model, variant, pred_json, checkpoint=None):
    """Register an existing non-empty prediction file without loading the detector."""
    pred_json = Path(pred_json)
    if not pred_json.is_file(): return False
    dependencies = [Path(checkpoint)] if checkpoint is not None else []
    dependencies.append(COCO_ANN["test"])
    if any(p.is_file() and p.stat().st_mtime > pred_json.stat().st_mtime for p in dependencies):
        print(f"[cache] saved prediction is older than its checkpoint or test annotations: {pred_json.name}")
        return False
    try:
        if not json.loads(pred_json.read_text()): return False
    except (OSError, json.JSONDecodeError):
        return False

    n_params, latency = None, None
    result_cache = OUTPUT_ROOT / "results_master.csv"
    if result_cache.is_file():
        cached = pd.read_csv(result_cache)
        hit = cached[(cached["model"] == model) & (cached["variant"] == variant)]
        if len(hit):
            row = hit.iloc[0]
            if pd.notna(row.get("params_M")): n_params = float(row["params_M"]) * 1e6
            if pd.notna(row.get("latency_ms")) and pd.notna(row.get("fps")):
                latency = {"latency_ms": float(row["latency_ms"]), "fps": float(row["fps"])}
    register_result(model, variant, pred_json, n_params=n_params, latency=latency)
    print(f"[reused] saved predictions for {model} / {variant}")
    return True

### 2.4 Ultralytics functions

The functions below add DOAM to Ultralytics model configurations, map pretrained weights, manage training and checkpoints, export predictions, and run the shared evaluation.

In [ ]:
import yaml
import ultralytics
from ultralytics import YOLO, RTDETR
from ultralytics.utils.torch_utils import intersect_dicts

ULTRA_CFG = Path(ultralytics.__file__).parent / "cfg" / "models"
DOAM_BASE_YAML = {"yolo26": ULTRA_CFG / "26" / "yolo26.yaml",
                  "rtdetr": ULTRA_CFG / "rt-detr" / "rtdetr-l.yaml"}
DOAM_OUT_YAML = {"yolo26": YOLO_DIR / "yolo26n-doam.yaml",
                 "rtdetr": YOLO_DIR / "rtdetr-l-doam.yaml"}

def register_doam_in_ultralytics():
    import ultralytics.nn.tasks as tasks
    for cls in (GatedConv2d, DOAM):
        cls.__module__ = "ultralytics.nn.tasks"
        setattr(tasks, cls.__name__, cls)

def _shift_from(value):
    if isinstance(value, int): return value + 1 if value >= 0 else value
    if isinstance(value, list): return [_shift_from(item) for item in value]
    return value

def make_doam_yaml(base_yaml, out_path):
    cfg = yaml.safe_load(Path(base_yaml).read_text(encoding="utf-8"))
    cfg["backbone"] = [[-1, 1, "DOAM", []]] + [list(layer) for layer in cfg["backbone"]]
    for section in ("backbone", "head"):
        for layer in cfg[section]: layer[0] = _shift_from(layer[0])
    Path(out_path).write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    return out_path

def load_pretrained_shifted(model, weights_path):
    checkpoint = torch.load(weights_path, map_location="cpu", weights_only=False)
    pretrained = checkpoint["model"] if isinstance(checkpoint, dict) else checkpoint
    remapped = {}
    for key, value in pretrained.float().state_dict().items():
        parts = key.split(".")
        if len(parts) >= 2 and parts[0] == "model" and parts[1].isdigit():
            parts[1] = str(int(parts[1]) + 1)
        remapped[".".join(parts)] = value
    matched = intersect_dicts(remapped, model.model.state_dict())
    model.model.load_state_dict(matched, strict=False)
    return len(matched), len(model.model.state_dict())

def build_ultra_doam(kind, base_weights, model_key):
    register_doam_in_ultralytics()
    constructor = YOLO if kind == "YOLO" else RTDETR
    yaml_path = make_doam_yaml(DOAM_BASE_YAML[model_key], DOAM_OUT_YAML[model_key])
    model = constructor(str(yaml_path))
    matched, total = load_pretrained_shifted(model, base_weights)
    print(f"[{model_key}+DOAM] loaded {matched}/{total} pretrained tensors")
    model.ckpt = {"model": model.model}
    return model

def ultra_predict_to_coco(model, split="test", out_path=None, conf=CONF_THRES, imgsz=IMGSZ):
    img_dir = YOLO_DIR / "images" / split
    coco = COCO(str(COCO_ANN[split]))
    valid_ids, predictions = set(coco.getImgIds()), []
    results = model.predict(source=str(img_dir), imgsz=imgsz, conf=conf, iou=IOU_NMS,
                            max_det=MAX_DET, device=(0 if DEVICE == "cuda" else "cpu"),
                            stream=True, verbose=False)
    for result in results:
        try: image_id = int(Path(result.path).stem)
        except ValueError: continue
        if image_id not in valid_ids or result.boxes is None: continue
        for box, score, cls in zip(result.boxes.xyxy.cpu().numpy(),
                                   result.boxes.conf.cpu().numpy(),
                                   result.boxes.cls.cpu().numpy().astype(int)):
            x1, y1, x2, y2 = map(float, box)
            predictions.append({"image_id": image_id, "category_id": int(cls),
                                "bbox": [x1, y1, x2 - x1, y2 - y1], "score": float(score)})
    out_path = Path(out_path or PRED_DIR / f"ultra_{split}.json")
    out_path.write_text(json.dumps(predictions))
    return out_path

def train_ultralytics(kind, weights, variant, model_key, run_training):
    constructor = YOLO if kind == "YOLO" else RTDETR
    run_name = f"{model_key}_{variant}"
    best = OUTPUT_ROOT / "ultra_runs" / run_name / "weights" / "best.pt"
    if variant == "doam": register_doam_in_ultralytics()
    if run_training:
        set_seed()
        model = build_ultra_doam(kind, weights, model_key) if variant == "doam" else constructor(weights)
        model.train(data=str(YOLO_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH[model_key],
                    device=(0 if DEVICE == "cuda" else "cpu"), seed=SEED,
                    project=str(OUTPUT_ROOT / "ultra_runs"), name=run_name, exist_ok=True,
                    pretrained=True, patience=max(5, EPOCHS // 6), plots=True, verbose=False)
    if not best.is_file(): raise FileNotFoundError(f"No checkpoint found for {run_name}: {best}")
    print("Loaded", best)
    return constructor(str(best))

def run_ultralytics_detector(display_name, kind, model_key, weights):
    for variant in ("baseline", "doam"):
        run_key = f"{model_key}_{variant}"
        best = OUTPUT_ROOT / "ultra_runs" / run_key / "weights" / "best.pt"
        pred_path = PRED_DIR / f"{run_key}_test.json"
        if not RUN_TRAINING[run_key]:
            if not best.is_file():
                raise FileNotFoundError(f"Training is disabled but checkpoint is missing: {best}")
            if reuse_saved_prediction(display_name, variant, pred_path, best): continue

        model = train_ultralytics(kind, weights, variant, model_key, RUN_TRAINING[run_key])
        pred = ultra_predict_to_coco(model, "test", pred_path)
        n_params = sum(parameter.numel() for parameter in model.model.parameters())
        sample_paths = sorted((YOLO_DIR / "images" / "test").iterdir())[:60]

        def predict_one(path, current_model=model):
            current_model.predict(source=str(path), imgsz=IMGSZ,
                                  device=(0 if DEVICE == "cuda" else "cpu"), verbose=False)

        latency = measure_latency(predict_one, sample_paths)
        register_result(display_name, variant, pred, n_params=n_params, latency=latency)
        del model
        if DEVICE == "cuda": torch.cuda.empty_cache()

def ultralytics_run_summary(model_key):
    config_rows, result_rows = [], []
    for variant in ("baseline", "doam"):
        run_name = f"{model_key}_{variant}"
        run_dir = OUTPUT_ROOT / "ultra_runs" / run_name
        args_path = run_dir / "args.yaml"
        results_path = run_dir / "results.csv"
        best_path = run_dir / "weights" / "best.pt"
        missing = [path.name for path in (args_path, results_path, best_path) if not path.is_file()]
        if missing:
            raise FileNotFoundError(f"Incomplete saved run for {run_name}: {', '.join(missing)}")

        args = yaml.safe_load(args_path.read_text(encoding="utf-8"))
        history = pd.read_csv(results_path)
        history.columns = history.columns.str.strip()
        map95_col = next(column for column in history if "map50-95" in column.lower())
        map50_col = next(column for column in history if "map50(" in column.lower())
        best_row = history.loc[history[map95_col].astype(float).idxmax()]

        config_rows.append({
            "variant": variant, "weights": Path(str(args["model"])).name,
            "imgsz": args["imgsz"], "batch": args["batch"], "epochs": args["epochs"],
            "optimizer_arg": args["optimizer"], "lr0_arg": args["lr0"],
            "weight_decay": args["weight_decay"], "cos_lr": args["cos_lr"],
            "patience": args["patience"], "pretrained": args["pretrained"],
            "freeze": "none" if args.get("freeze") is None else args["freeze"],
        })
        result_rows.append({
            "variant": variant, "completed_epochs": len(history),
            "best_epoch": int(best_row["epoch"]),
            "best_val_mAP50": float(best_row[map50_col]),
            "best_val_mAP50_95": float(best_row[map95_col]),
            "training_minutes": float(history["time"].iloc[-1]) / 60,
            "checkpoint_MiB": best_path.stat().st_size / (1024 ** 2),
        })

    config = pd.DataFrame(config_rows).set_index("variant")
    results = pd.DataFrame(result_rows).set_index("variant")
    return config, results

---
## Section 3: Faster R-CNN

This section uses torchvision `fasterrcnn_resnet50_fpn` with COCO pretrained weights. The box predictor is replaced for the five OPIXray classes and the background class.

The baseline receives the original image and uses early stopping after the first learning-rate decay. The DOAM variant processes the image before detection and trains for the number of epochs completed by the baseline. Both variants validate after each epoch and save their own checkpoint with the highest validation `mAP50_95`.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from tqdm import tqdm

def letterbox_params(ow, oh, size):
    scale = min(size / ow, size / oh)
    nw, nh = round(ow * scale), round(oh * scale)
    return scale, (size - nw) // 2, (size - nh) // 2, nw, nh

_IMG_CACHE = {}

class CocoDetDataset(Dataset):
    """COCO dataset with letterbox resizing for Faster R-CNN."""
    def __init__(self, split, img_size=IMGSZ):
        self.coco = COCO(str(COCO_ANN[split]))
        self.img_dir = COCO_IMG[split]
        self.ids = list(self.coco.imgs.keys())
        self.img_size = img_size
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        img_id = self.ids[idx]
        info = self.coco.loadImgs(img_id)[0]
        ow, oh = info["width"], info["height"]
        scale, pad_x, pad_y, nw, nh = letterbox_params(ow, oh, self.img_size)
        key = (img_id, self.img_size)
        img = _IMG_CACHE.get(key) if CACHE_IMAGES else None
        if img is None:
            src = Image.open(self.img_dir / info["file_name"]).convert("RGB").resize((nw, nh))
            img = Image.new("RGB", (self.img_size, self.img_size))
            img.paste(src, (pad_x, pad_y))
            if CACHE_IMAGES:
                _IMG_CACHE[key] = img
        boxes, labels = [], []
        for a in self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id)):
            x, y, w, h = a["bbox"]
            boxes.append([x*scale+pad_x, y*scale+pad_y, (x+w)*scale+pad_x, (y+h)*scale+pad_y])
            labels.append(a["category_id"] + 1)
        target = {"boxes": torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),
                  "labels": torch.as_tensor(labels, dtype=torch.int64),
                  "image_id": torch.tensor([img_id])}
        return TF.to_tensor(img), target

def collate_fn(batch): return tuple(zip(*batch))

def build_faster_rcnn(front_kind="baseline", pretrained=True):
    set_seed()
    weights = "DEFAULT" if pretrained else None
    model = fasterrcnn_resnet50_fpn(weights=weights, weights_backbone=None, min_size=IMGSZ, max_size=IMGSZ,
                                    box_score_thresh=0.001, box_detections_per_img=MAX_DET)
    in_feat = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, NUM_CLASSES + 1)
    return DetectorWithFrontEnd(model, build_front(front_kind)).to(DEVICE)

In [ ]:
@torch.no_grad()
def frcnn_predict_to_coco(model, split="test", out_path=None, conf=CONF_THRES):
    """Run inference and dump COCO-format predictions in ORIGINAL image pixels."""
    model.eval()
    ds = CocoDetDataset(split)
    loader = DataLoader(ds, batch_size=BATCH["faster_rcnn"], shuffle=False,
                        collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
    preds = []
    for images, targets in tqdm(loader, desc=f"predict[{split}]", leave=False):
        images = [im.to(DEVICE) for im in images]
        outputs = model(images)
        for tgt, out in zip(targets, outputs):
            img_id = int(tgt["image_id"])
            info = ds.coco.loadImgs(img_id)[0]
            scale, pad_x, pad_y, _, _ = letterbox_params(info["width"], info["height"], IMGSZ)
            for box, score, label in zip(out["boxes"].cpu().numpy(),
                                         out["scores"].cpu().numpy(),
                                         out["labels"].cpu().numpy()):
                if score < conf: continue
                x1, y1, x2, y2 = box
                ox1, oy1 = (x1 - pad_x) / scale, (y1 - pad_y) / scale
                ox2, oy2 = (x2 - pad_x) / scale, (y2 - pad_y) / scale
                preds.append({"image_id": img_id, "category_id": int(label) - 1,
                              "bbox": [float(ox1), float(oy1), float(ox2 - ox1), float(oy2 - oy1)],
                              "score": float(score)})
    out_path = out_path or (PRED_DIR / f"faster_rcnn_{split}.json")
    Path(out_path).write_text(json.dumps(preds))
    return out_path

def train_faster_rcnn(front_kind, epochs=EPOCHS):
    """Train a Faster R-CNN variant or load its saved checkpoint."""
    run_key = f"faster_rcnn_{front_kind}"
    ckpt = CKPT_DIR / f"{run_key}.pth"
    history_path = CKPT_DIR / f"{run_key}_history.csv"
    run_training = RUN_TRAINING[run_key]
    train_epochs = epochs
    if run_training and front_kind == "doam":
        baseline_history = CKPT_DIR / "faster_rcnn_baseline_history.csv"
        if not baseline_history.is_file():
            raise FileNotFoundError("DOAM training requires the baseline training history")
        baseline_df = pd.read_csv(baseline_history)
        required = {"train_seed", "scheduler_step"}
        if not required.issubset(baseline_df.columns):
            raise ValueError("Rerun the baseline with the current paired training logic before training DOAM")
        if not (baseline_df["train_seed"] == SEED).all():
            raise ValueError("The baseline history uses a different training seed")
        if not (baseline_df["scheduler_step"] == max(1, EPOCHS // 3)).all():
            raise ValueError("The baseline history uses a different learning-rate schedule")
        train_epochs = len(baseline_df)
        if train_epochs < 1:
            raise ValueError("The baseline training history contains no epochs")
        print(f"[frcnn/doam] paired training budget: {train_epochs} epochs")
    model = build_faster_rcnn(front_kind, pretrained=run_training)
    if not run_training:
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        print("Loaded", ckpt.name)
        return model

    train_generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(CocoDetDataset("train"), batch_size=BATCH["faster_rcnn"], shuffle=True,
                              collate_fn=collate_fn, num_workers=NUM_WORKERS,
                              persistent_workers=(NUM_WORKERS > 0), pin_memory=True,
                              generator=train_generator)
    params = [p for p in model.parameters() if p.requires_grad]
    base_lr = 0.00125 * BATCH["faster_rcnn"]
    optim = torch.optim.SGD(params, lr=base_lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.StepLR(optim, step_size=max(1, EPOCHS // 3), gamma=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
    warmup_iters = min(500, max(1, len(train_loader) - 1))

    best_map, early_stop_best = -1.0, -1.0
    no_improve_checks, gstep, history = 0, 0, []
    for ep in range(train_epochs):
        model.train(); running = 0.0
        for images, targets in tqdm(train_loader, desc=f"train ep{ep+1}/{train_epochs}", leave=False):
            if gstep < warmup_iters:
                for group in optim.param_groups:
                    group["lr"] = base_lr * (0.1 + 0.9 * gstep / warmup_iters)
            images = [im.to(DEVICE, non_blocking=True) for im in images]
            targets = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for t in targets]
            optim.zero_grad()
            with torch.autocast("cuda", enabled=(DEVICE == "cuda")):
                loss = sum(model(images, targets).values())
            scaler.scale(loss).backward(); scaler.step(optim); scaler.update()
            running += loss.item(); gstep += 1

        sched.step()
        train_loss = running / len(train_loader)
        vp = frcnn_predict_to_coco(model, "val", PRED_DIR / f"{run_key}_val.json")
        vm = coco_eval(vp, COCO_ANN["val"])
        vmap, vmap50 = vm["mAP50_95"], vm["mAP50"]
        print(f"[frcnn/{front_kind}] epoch {ep+1}/{train_epochs}  loss={train_loss:.4f}  val mAP50-95={vmap:.4f}")

        if vmap > best_map:
            best_map = vmap
            torch.save(model.state_dict(), ckpt)
            print("  saved best checkpoint")
        if front_kind == "baseline":
            if vmap > early_stop_best + FRCNN_MIN_DELTA:
                early_stop_best, no_improve_checks = vmap, 0
            else:
                no_improve_checks += 1

        history.append({"epoch": ep + 1, "train_loss": train_loss,
                        "val_map50": vmap50, "val_map50_95": vmap,
                        "train_seed": SEED, "scheduler_step": max(1, EPOCHS // 3)})
        pd.DataFrame(history).to_csv(history_path, index=False)
        if (front_kind == "baseline" and ep + 1 >= FRCNN_MIN_EPOCHS
                and no_improve_checks >= FRCNN_PATIENCE):
            print(f"[frcnn/{front_kind}] early stopping at epoch {ep+1}; "
                  f"no mAP50-95 improvement >= {FRCNN_MIN_DELTA} for {FRCNN_PATIENCE} checks")
            break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return model

In [ ]:
# Run the Faster R-CNN baseline and DOAM variants.
for v in ("baseline", "doam"):
    run_key = f"faster_rcnn_{v}"
    saved_ckpt = CKPT_DIR / f"{run_key}.pth"
    saved_pred = PRED_DIR / f"{run_key}_test.json"
    if not RUN_TRAINING[run_key]:
        if not saved_ckpt.is_file():
            raise FileNotFoundError(f"Training is disabled but checkpoint is missing: {saved_ckpt}")
        if reuse_saved_prediction("Faster R-CNN", v, saved_pred, saved_ckpt):
            continue
    model = train_faster_rcnn(v)
    pred = frcnn_predict_to_coco(model, "test", saved_pred)
    n_params = sum(p.numel() for p in model.parameters())
    ds = CocoDetDataset("test")
    sample_paths = [ds.img_dir / ds.coco.loadImgs(i)[0]["file_name"] for i in ds.ids[:60]]
    def _predict_one(p, _m=model):
        img = Image.open(p).convert("RGB").resize((IMGSZ, IMGSZ))
        with torch.no_grad():
            _m.eval(); _m([TF.to_tensor(img).to(DEVICE)])
    lat = measure_latency(_predict_one, sample_paths)
    register_result("Faster R-CNN", v, pred, n_params=n_params, latency=lat)
    del model
    if DEVICE == "cuda": torch.cuda.empty_cache()

---
## Section 4: YOLO26

This section uses Ultralytics YOLO26n with the `yolo26n.pt` COCO pretrained weights and the shared OPIXray split. Its backbone and detection head produce predictions from the P3, P4, and P5 feature levels.

The DOAM variant adds DOAM as the first model layer. Both variants export test predictions to the COCO format used by the shared evaluator.

### 4.1 Unified baseline and DOAM

Both variants use the shared training and evaluation functions from Section 2. Their training settings remain identical except for the DOAM input module.

In [ ]:
YOLO_YAML = YOLO_DIR / "dataset_unified.yaml"
YOLO_YAML.write_text(yaml.safe_dump({
    "path": str(YOLO_DIR.resolve()),
    "train": "images/train", "val": "images/val", "test": "images/test",
    "nc": NUM_CLASSES, "names": {i: n for i, n in enumerate(CLASS_NAMES)},
}, sort_keys=False, allow_unicode=True))
print(YOLO_YAML.read_text())

YOLO26_WEIGHTS = "yolo26n.pt"

In [ ]:
run_ultralytics_detector("YOLO26", "YOLO", "yolo26", YOLO26_WEIGHTS)

### 4.2 Saved training record

The first table reads the training arguments saved by Ultralytics. The second table reports the completed epochs, best validation epoch, training time, and checkpoint size. Test metrics are calculated with the shared evaluator in Section 6.

In [ ]:
yolo26_config, yolo26_training = ultralytics_run_summary("yolo26")
display(yolo26_config)
display(yolo26_training.round(4))

### 4.3 Model-specific training

This configuration records the YOLO26n training setup used in the separate model notebook. It keeps the same pretrained model and input size while defining YOLO-specific augmentation and checkpoint settings. The run has its own switch and output directory.

In [ ]:
YOLO26_MODEL_SPECIFIC_NAME = "yolo26_model_specific"
YOLO26_MODEL_SPECIFIC_ROOT = OUTPUT_ROOT / "model_specific_runs"
YOLO26_MODEL_SPECIFIC_ARGS = {
    "epochs": 30,
    "imgsz": 640,
    "batch": 8,
    "workers": 2,
    "pretrained": True,
    "patience": 4,
    "save": True,
    "save_period": 10,
    "plots": True,
    "cache": False,
    "degrees": 5.0,
    "translate": 0.1,
    "scale": 0.4,
    "fliplr": 0.5,
    "flipud": 0.0,
    "mosaic": 1.0,
    "mixup": 0.0,
    "seed": SEED,
}

if RUN_TRAINING["yolo26_model_specific"]:
    yolo26_specific = YOLO(YOLO26_WEIGHTS)
    yolo26_specific.train(
        data=str(YOLO_YAML),
        device=(0 if DEVICE == "cuda" else "cpu"),
        project=str(YOLO26_MODEL_SPECIFIC_ROOT),
        name=YOLO26_MODEL_SPECIFIC_NAME,
        exist_ok=True,
        **YOLO26_MODEL_SPECIFIC_ARGS,
    )
    del yolo26_specific
    if DEVICE == "cuda": torch.cuda.empty_cache()

YOLO26_MODEL_SPECIFIC_CKPT = (YOLO26_MODEL_SPECIFIC_ROOT / YOLO26_MODEL_SPECIFIC_NAME
                                / "weights" / "best.pt")
YOLO26_MODEL_SPECIFIC_PRED = PRED_DIR / "yolo26_model_specific_test.json"
display(pd.Series(YOLO26_MODEL_SPECIFIC_ARGS, name="YOLO26 model-specific configuration"))

if YOLO26_MODEL_SPECIFIC_CKPT.is_file():
    if not reuse_saved_prediction("YOLO26", "model_specific",
                                  YOLO26_MODEL_SPECIFIC_PRED, YOLO26_MODEL_SPECIFIC_CKPT):
        yolo26_specific = YOLO(str(YOLO26_MODEL_SPECIFIC_CKPT))
        pred = ultra_predict_to_coco(
            yolo26_specific, "test", YOLO26_MODEL_SPECIFIC_PRED,
            imgsz=YOLO26_MODEL_SPECIFIC_ARGS["imgsz"],
        )
        n_params = sum(parameter.numel() for parameter in yolo26_specific.model.parameters())
        sample_paths = sorted((YOLO_DIR / "images" / "test").iterdir())[:60]

        def predict_yolo26_specific(path):
            yolo26_specific.predict(
                source=str(path), imgsz=YOLO26_MODEL_SPECIFIC_ARGS["imgsz"],
                device=(0 if DEVICE == "cuda" else "cpu"), verbose=False,
            )

        latency = measure_latency(predict_yolo26_specific, sample_paths)
        register_result("YOLO26", "model_specific", pred,
                        n_params=n_params, latency=latency)
        del yolo26_specific
        if DEVICE == "cuda": torch.cuda.empty_cache()
else:
    print("Model-specific checkpoint: not created")

---
## Section 5: RT-DETR

This section uses Ultralytics RT-DETR-L with the `rtdetr-l.pt` COCO pretrained weights and the shared OPIXray split. The model combines a convolutional backbone, multi-scale feature fusion, an AIFI module, and a transformer decoder.

The DOAM variant uses the shared model configuration functions from Section 2. Both variants export test predictions to the COCO format used by the shared evaluator.

### 5.1 Unified baseline and DOAM

The shared runner applies the unified input size, epoch limit, checkpoint handling, prediction export, and COCO evaluation to the RT-DETR baseline and DOAM variants.

In [ ]:
RTDETR_WEIGHTS = "rtdetr-l.pt"
run_ultralytics_detector("RT-DETR", "RTDETR", "rtdetr", RTDETR_WEIGHTS)

### 5.2 Saved training record

The saved arguments identify the settings used by each checkpoint, including the optimizer, learning rate, patience, and backbone freeze state. The training table uses the highest validation `mAP50-95` epoch from each `results.csv` file.

In [ ]:
rtdetr_config, rtdetr_training = ultralytics_run_summary("rtdetr")
display(rtdetr_config)
display(rtdetr_training.round(4))

### 5.3 Model-specific training

This independent RT-DETR configuration uses the settings from the separate model notebook. It uses AdamW with a lower initial learning rate, cosine scheduling, and longer early-stopping patience.

In [ ]:
RTDETR_MODEL_SPECIFIC_NAME = "rtdetr_model_specific"
RTDETR_MODEL_SPECIFIC_ROOT = OUTPUT_ROOT / "model_specific_runs"
RTDETR_MODEL_SPECIFIC_ARGS = {
    "epochs": 30,
    "imgsz": 512,
    "batch": 6,
    "workers": 2,
    "pretrained": True,
    "optimizer": "AdamW",
    "lr0": 1e-4,
    "weight_decay": 1e-4,
    "cos_lr": True,
    "patience": 15,
    "deterministic": False,
    "amp": True,
    "save": True,
    "save_period": 10,
    "plots": True,
    "seed": SEED,
}

if RUN_TRAINING["rtdetr_model_specific"]:
    rtdetr_specific = RTDETR(RTDETR_WEIGHTS)
    rtdetr_specific.train(
        data=str(YOLO_YAML),
        device=(0 if DEVICE == "cuda" else "cpu"),
        project=str(RTDETR_MODEL_SPECIFIC_ROOT),
        name=RTDETR_MODEL_SPECIFIC_NAME,
        exist_ok=True,
        **RTDETR_MODEL_SPECIFIC_ARGS,
    )
    del rtdetr_specific
    if DEVICE == "cuda": torch.cuda.empty_cache()

RTDETR_MODEL_SPECIFIC_CKPT = (RTDETR_MODEL_SPECIFIC_ROOT / RTDETR_MODEL_SPECIFIC_NAME
                                / "weights" / "best.pt")
RTDETR_MODEL_SPECIFIC_PRED = PRED_DIR / "rtdetr_model_specific_test.json"
display(pd.Series(RTDETR_MODEL_SPECIFIC_ARGS, name="RT-DETR model-specific configuration"))

if RTDETR_MODEL_SPECIFIC_CKPT.is_file():
    if not reuse_saved_prediction("RT-DETR", "model_specific",
                                  RTDETR_MODEL_SPECIFIC_PRED, RTDETR_MODEL_SPECIFIC_CKPT):
        rtdetr_specific = RTDETR(str(RTDETR_MODEL_SPECIFIC_CKPT))
        pred = ultra_predict_to_coco(
            rtdetr_specific, "test", RTDETR_MODEL_SPECIFIC_PRED,
            imgsz=RTDETR_MODEL_SPECIFIC_ARGS["imgsz"],
        )
        n_params = sum(parameter.numel() for parameter in rtdetr_specific.model.parameters())
        sample_paths = sorted((YOLO_DIR / "images" / "test").iterdir())[:60]

        def predict_rtdetr_specific(path):
            rtdetr_specific.predict(
                source=str(path), imgsz=RTDETR_MODEL_SPECIFIC_ARGS["imgsz"],
                device=(0 if DEVICE == "cuda" else "cpu"), verbose=False,
            )

        latency = measure_latency(predict_rtdetr_specific, sample_paths)
        register_result("RT-DETR", "model_specific", pred,
                        n_params=n_params, latency=latency)
        del rtdetr_specific
        if DEVICE == "cuda": torch.cuda.empty_cache()
else:
    print("Model-specific checkpoint: not created")

---
## Section 6: Results and Evaluation

Every row below is produced by the same `pycocotools` evaluator on the same test set. Each model and variant is measured from its exported predictions. YOLO26 and RT-DETR include a `model_specific` row after their independent training runs produce a checkpoint.

In [ ]:
results_df = pd.DataFrame(RESULTS)
if len(results_df):
    cols = ["model", "variant", "mAP50", "mAP50_95", "recall", "params_M", "latency_ms", "fps"]
    variant_order = {"baseline": 0, "model_specific": 1, "doam": 2}
    main = results_df[cols].assign(
        _variant_order=results_df["variant"].map(variant_order).fillna(99)
    ).sort_values(["model", "_variant_order"]).drop(columns="_variant_order").reset_index(drop=True)
    main.to_csv(OUTPUT_ROOT / "results_master.csv", index=False)
    display(main.round(4))
else:
    print("No results yet - train at least one model (set RUN_TRAINING[...] = True).")

In [ ]:
# Overall mAP bar chart + accuracy/speed trade-off
if len(results_df):
    dfp = results_df.copy()
    variant_order = {"baseline": 0, "model_specific": 1, "doam": 2}
    dfp["_variant_order"] = dfp["variant"].map(variant_order).fillna(99)
    dfp = dfp.sort_values(["model", "_variant_order"])
    dfp["label"] = dfp["model"] + "\n" + dfp["variant"].str.replace("_", " ")
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    ax[0].bar(dfp["label"], dfp["mAP50"]); ax[0].set_title("mAP@0.5 by model / variant")
    ax[0].set_ylabel("mAP@0.5"); ax[0].tick_params(axis="x", rotation=30)
    sub = dfp.dropna(subset=["fps"])
    if len(sub):
        ax[1].scatter(sub["fps"], sub["mAP50"])
        for _, r in sub.iterrows(): ax[1].annotate(r["label"].replace("\n","/"), (r["fps"], r["mAP50"]))
        ax[1].set_xlabel("FPS"); ax[1].set_ylabel("mAP@0.5"); ax[1].set_title("Accuracy vs. speed trade-off")
    plt.tight_layout(); plt.savefig(FIG_DIR / "results_overview.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# --- Per-class AP@0.5 ---
if len(results_df):
    pc_cols = [f"AP50_{c}" for c in CLASS_NAMES]
    variant_order = {"baseline": 0, "model_specific": 1, "doam": 2}
    ordered = results_df.assign(
        _variant_order=results_df["variant"].map(variant_order).fillna(99)
    ).sort_values(["model", "_variant_order"])
    pc = ordered.set_index(ordered["model"] + "/" + ordered["variant"].str.replace("_", " "))[pc_cols]
    pc.columns = CLASS_NAMES
    ax = pc.T.plot(kind="bar", figsize=(12, 5))
    ax.set_title("Per-class AP@0.5"); ax.set_ylabel("AP@0.5"); ax.set_xlabel("Class")
    plt.xticks(rotation=20, ha="right"); plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout(); plt.savefig(FIG_DIR / "results_per_class.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Baseline vs +DOAM paired comparison
if len(results_df):
    rows = []
    for model in results_df["model"].unique():
        b = results_df[(results_df.model == model) & (results_df.variant == "baseline")]
        d = results_df[(results_df.model == model) & (results_df.variant == "doam")]
        if len(b) and len(d):
            b0, d0 = float(b["mAP50"].iloc[0]), float(d["mAP50"].iloc[0])
            rows.append({"model": model, "baseline_mAP50": b0, "doam_mAP50": d0,
                         "delta_pp": (d0 - b0) * 100,
                         "relative_%": (d0 - b0) / b0 * 100 if b0 > 0 else np.nan})
    if rows:
        paired = pd.DataFrame(rows)
        print("DOAM effect on mAP@0.5 (percentage points and relative %):")
        display(paired.round(3))
    else:
        print("DOAM variants not trained yet - paired comparison pending (TODO).")

### 6.1 Per-model diagnostic figures

Training convergence, loss curves, precision-recall curves and per-model confusion matrices, computed from the same runs and predictions used for the results table.

In [ ]:
# Validation mAP@0.5 over epochs. Ultralytics from results.csv; Faster R-CNN from its saved history.
fig, ax = plt.subplots(figsize=(9, 5))
ultra_histories = [
    ("YOLO26 baseline", OUTPUT_ROOT / "ultra_runs" / "yolo26_baseline" / "results.csv"),
    ("YOLO26 model specific", OUTPUT_ROOT / "model_specific_runs" / "yolo26_model_specific" / "results.csv"),
    ("YOLO26 +DOAM", OUTPUT_ROOT / "ultra_runs" / "yolo26_doam" / "results.csv"),
    ("RT-DETR baseline", OUTPUT_ROOT / "ultra_runs" / "rtdetr_baseline" / "results.csv"),
    ("RT-DETR model specific", OUTPUT_ROOT / "model_specific_runs" / "rtdetr_model_specific" / "results.csv"),
    ("RT-DETR +DOAM", OUTPUT_ROOT / "ultra_runs" / "rtdetr_doam" / "results.csv"),
]
for label, p in ultra_histories:
    if p.exists():
        h = pd.read_csv(p); h.columns = [c.strip() for c in h.columns]
        mc = [c for c in h.columns if "map50(" in c.lower()]
        ec = "epoch" if "epoch" in h.columns else h.columns[0]
        if mc: ax.plot(h[ec], h[mc[0]] * 100, label=label)
for label, kind in [("FRCNN baseline", "baseline"), ("FRCNN +DOAM", "doam")]:
    p = CKPT_DIR / f"faster_rcnn_{kind}_history.csv"
    if p.exists():
        h = pd.read_csv(p).dropna(subset=["val_map50"])
        if len(h): ax.plot(h["epoch"], h["val_map50"] * 100, "--o", label=label)
ax.set_title("Validation mAP@0.5 over training epochs"); ax.set_xlabel("epoch"); ax.set_ylabel("mAP@0.5 (%)")
ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG_DIR / "training_curves.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Training and validation loss curves

LOSS_RUNS = {
    "Faster R-CNN": [("baseline",       CKPT_DIR / "faster_rcnn_baseline_history.csv"),
                     ("+DOAM",          CKPT_DIR / "faster_rcnn_doam_history.csv")],
    "YOLO26":       [("baseline",       OUTPUT_ROOT / "ultra_runs" / "yolo26_baseline" / "results.csv"),
                     ("model specific", OUTPUT_ROOT / "model_specific_runs" / "yolo26_model_specific" / "results.csv"),
                     ("+DOAM",          OUTPUT_ROOT / "ultra_runs" / "yolo26_doam" / "results.csv")],
    "RT-DETR":      [("baseline",       OUTPUT_ROOT / "ultra_runs" / "rtdetr_baseline" / "results.csv"),
                     ("model specific", OUTPUT_ROOT / "model_specific_runs" / "rtdetr_model_specific" / "results.csv"),
                     ("+DOAM",          OUTPUT_ROOT / "ultra_runs" / "rtdetr_doam" / "results.csv")],
}
VARIANT_COLOR = {"baseline": "#1f77b4", "model specific": "#ff7f0e", "+DOAM": "#d62728"}

def load_loss_history(path):
    """(epochs, train_total, val_total, component_names) for one run. val_total is None
    when only training loss was recorded (torchvision returns no loss in eval mode)."""
    h = pd.read_csv(path); h.columns = [c.strip() for c in h.columns]
    if "train_loss" in h.columns:                                    # Faster R-CNN history
        return h["epoch"], h["train_loss"], None, ["RPN + ROI"]
    tr = [c for c in h.columns if c.startswith("train/") and c.endswith("_loss")]
    va = [c for c in h.columns if c.startswith("val/") and c.endswith("_loss")]
    comps = [c.split("/")[1].replace("_loss", "") for c in tr]
    return h["epoch"], h[tr].sum(axis=1), (h[va].sum(axis=1) if va else None), comps

fig, axes = plt.subplots(2, len(LOSS_RUNS), figsize=(5.6 * len(LOSS_RUNS), 8.6))
rows = []
for col, (model, runs) in enumerate(LOSS_RUNS.items()):
    full, zoom = axes[0][col], axes[1][col]
    comps, curves = [], []
    for variant, path in runs:
        if not Path(path).is_file():
            print(f"[skip] no training history for {model} / {variant}: {path}")
            continue
        ep, tr, va, comps = load_loss_history(path)
        colour = VARIANT_COLOR[variant]
        curves.append((variant, colour, ep, tr, va))
        for ax in (full, zoom):
            ax.plot(ep, tr, color=colour, linewidth=1.8, label=f"{variant} train")
            if va is not None:
                ax.plot(ep, va, color=colour, linewidth=1.8, linestyle="--", label=f"{variant} val")
        if va is not None:
            best = int(np.argmin(va.values))
            zoom.plot(ep.iloc[best], va.iloc[best], "o", color=colour, markersize=7,
                      markerfacecolor="white", markeredgewidth=1.8)
        rows.append({"model": model, "variant": variant, "epochs": int(ep.iloc[-1]),
                     "final_train_loss": float(tr.iloc[-1]),
                     "best_val_loss": float(va.min()) if va is not None else np.nan,
                     "best_val_epoch": int(ep.iloc[int(np.argmin(va.values))]) if va is not None else np.nan,
                     "val_minus_train_end": float(va.iloc[-1] - tr.iloc[-1]) if va is not None else np.nan})

    full.set_yscale("log")
    full.set_title(f"{model}\nloss = {' + '.join(comps)}", fontsize=11)
    full.set_ylabel("loss, log scale")

    # tail window: skip the initial drop so the variants are readable
    last = max(int(ep.iloc[-1]) for _, _, ep, _, _ in curves)
    cut = max(2, int(round(last * 0.35)))
    tail = []
    for _, _, e, tr_, va_ in curves:
        mask = (e >= cut).values
        tail.append(tr_[mask])
        if va_ is not None:
            tail.append(va_[mask])
    tail = [s for s in tail if len(s)]
    lo, hi = min(s.min() for s in tail), max(s.max() for s in tail)
    pad = (hi - lo) * 0.12 or abs(hi) * 0.05
    zoom.set_xlim(cut, last + 0.5); zoom.set_ylim(lo - pad, hi + pad)
    zoom.set_title(f"{model}: converged tail (epoch >= {cut})", fontsize=11)
    zoom.set_ylabel("loss")

    for ax in (full, zoom):
        ax.set_xlabel("epoch"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
    if not any(va_ is not None for _, _, _, _, va_ in curves):
        full.text(0.98, 0.95, "torchvision reports no validation loss;\nvalidation mAP is in the figure above",
                  transform=full.transAxes, ha="right", va="top", fontsize=8, color="#555555")

fig.suptitle("Loss curves per detector: training (solid) and validation (dashed), one colour per variant."
             "\nCircle marks the lowest validation loss. Loss definitions differ between panels, so compare within a panel.",
             fontsize=12)
plt.tight_layout(rect=(0, 0, 1, 0.95))
plt.savefig(FIG_DIR / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

loss_summary = pd.DataFrame(rows)
display(loss_summary.round(4))

**What the loss curves show:**
- Faster R-CNN baseline and +DOAM are indistinguishable (final training loss 0.0477 against 0.0478): the front module changes almost nothing about how this detector optimises, which is consistent with its small test difference.
- YOLO26 +DOAM reaches the lowest validation loss of the three YOLO variants (2.648 at epoch 28, against 2.684 for the baseline), the one setting where DOAM also improves test mAP.
- RT-DETR model specific reaches by far the lowest training loss (1.153) but the highest validation loss (1.226), so its heavy augmentation setting overfits relative to the shared recipe. That matches its 2.71pp mAP drop, and it is the clearest train/validation divergence in the figure.
- Loss is measured differently by each family (box + cls + dfl, giou + cls + l1, RPN + ROI), so the comparison is only meaningful inside one panel.

In [ ]:
# Precision-Recall @ IoU=0.5, averaged over classes, from the unified predictions.
def pr_curve(pred_json):
    preds = json.loads(Path(pred_json).read_text())
    if not preds: return None, None
    gt = COCO(str(COCO_ANN["test"])); dt = gt.loadRes(preds); ev = COCOeval(gt, dt, iouType="bbox")
    with contextlib.redirect_stdout(io.StringIO()): ev.evaluate(); ev.accumulate()
    prec = ev.eval["precision"][0, :, :, 0, -1].copy(); prec[prec < 0] = float("nan")
    return ev.params.recThrs, np.nanmean(prec, axis=1)

if len(results_df):
    fig, ax = plt.subplots(figsize=(7, 6))
    for (model, variant), pred in sorted(PRED_STORE.items()):
        r, p = pr_curve(pred)
        if r is not None: ax.plot(r, p, label=f"{model} {variant}")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title("Precision-Recall @ IoU=0.5")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(FIG_DIR / "pr_curves.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Confusion matrix 
def confusion_for(model, variant, score_thr=0.3, iou_thr=0.5):
    key = (model, variant)
    if key not in PRED_STORE: return None
    preds = json.loads(Path(PRED_STORE[key]).read_text())
    if not preds: return None
    coco = COCO(str(COCO_ANN["test"]))
    by_img = defaultdict(list)
    for p in preds:
        if p["score"] >= score_thr: by_img[p["image_id"]].append(p)
    n = NUM_CLASSES
    cm = np.zeros((n + 1, n + 1)) 
    def iou(a, b):
        ax1, ay1, aw, ah = a; bx1, by1, bw, bh = b
        ax2, ay2, bx2, by2 = ax1+aw, ay1+ah, bx1+bw, by1+bh
        ix1, iy1, ix2, iy2 = max(ax1,bx1), max(ay1,by1), min(ax2,bx2), min(ay2,by2)
        inter = max(0, ix2-ix1) * max(0, iy2-iy1)
        union = aw*ah + bw*bh - inter
        return inter/union if union > 0 else 0
    for img_id in coco.getImgIds():
        gts = coco.loadAnns(coco.getAnnIds(imgIds=img_id))
        dets = by_img.get(img_id, [])
        used = set()
        for g in gts:
            best, bi = iou_thr, -1
            for i, d in enumerate(dets):
                if i in used: continue
                j = iou(g["bbox"], d["bbox"])
                if j >= best: best, bi = j, i
            if bi >= 0:
                cm[g["category_id"], dets[bi]["category_id"]] += 1; used.add(bi)
            else:
                cm[g["category_id"], n] += 1                     # missed (false negative)
        for i, d in enumerate(dets):
            if i not in used: cm[n, d["category_id"]] += 1       # false positive
    return cm

def _draw_cm(cm, ax, title):
    cmn = np.nan_to_num(cm / cm.sum(axis=1, keepdims=True))
    labels = CLASS_NAMES + ["(bg)"]
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=7)
    ax.set_ylabel("Ground truth"); ax.set_xlabel("Predicted"); ax.set_title(title, fontsize=9)
    for i in range(cmn.shape[0]):
        for j in range(cmn.shape[1]):
            if cmn[i, j] > 0.005:
                ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center",
                        color="white" if cmn[i, j] > 0.5 else "black", fontsize=6)

if len(results_df):
    _cm_models = [m for m in ["Faster R-CNN", "YOLO26", "RT-DETR"] if (m, "baseline") in PRED_STORE]
    if _cm_models:
        _cm_variants = ["baseline", "model_specific", "doam"]
        fig, axes = plt.subplots(len(_cm_variants), len(_cm_models),
                                 figsize=(6 * len(_cm_models), 5 * len(_cm_variants)), squeeze=False)
        for col, m in enumerate(_cm_models):
            for row, variant in enumerate(_cm_variants):
                ax = axes[row, col]
                cm = confusion_for(m, variant) if (m, variant) in PRED_STORE else None
                if cm is not None:
                    _draw_cm(cm, ax, f"{m} {variant}")
                else:
                    ax.axis("off")
        plt.tight_layout(); plt.savefig(FIG_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight"); plt.show()
    else:
        print("No non-empty predictions to build confusion matrices yet.")

### 6.2 Occlusion-level analysis (OL1 / OL2 / OL3)

DOAM is intended to improve robustness under occlusion, so we report mAP separately for the three occlusion levels. OPIXray provides this split as `OPIXray/test/test_knife-{1,2,3}.txt` (OL1=922, OL2=548, OL3=306 images). We read it directly and score each level with the same evaluator by restricting `img_ids`. The question this answers: does DOAM's gain increase from OL1 to OL3?

In [ ]:
def eval_by_occlusion():
    # Occlusion levels ship with OPIXray: OPIXray/test/test_knife-{1,2,3}.txt = OL1 / OL2 / OL3.
    ol_files = {"OL1": OPIXRAY_RAW / "test" / "test_knife-1.txt",
                "OL2": OPIXRAY_RAW / "test" / "test_knife-2.txt",
                "OL3": OPIXRAY_RAW / "test" / "test_knife-3.txt"}
    if not all(p.exists() for p in ol_files.values()):
        print("[skip] OPIXray occlusion lists not found under", OPIXRAY_RAW / "test")
        return None
    def read_ol(p):
        return [int(line.strip().split()[0].split(".")[0])
                for line in p.read_text().splitlines() if line.strip()]
    ol_ids = {k: read_ol(p) for k, p in ol_files.items()}
    print("Occlusion-level image counts:", {k: len(v) for k, v in ol_ids.items()})
    rows = []
    for (model, variant), pred in PRED_STORE.items():
        for lvl, ids in ol_ids.items():
            m = coco_eval(pred, COCO_ANN["test"], img_ids=ids)
            rows.append({"model": model, "variant": variant, "OL": lvl,
                         "mAP50": m["mAP50"], "mAP50_95": m["mAP50_95"]})
    return pd.DataFrame(rows)

ol_df = eval_by_occlusion()
if ol_df is not None:
    ol_df["mAP50_pct"] = ol_df["mAP50"] * 100
    piv = ol_df.pivot_table(index=["model", "variant"], columns="OL", values="mAP50_pct")[["OL1", "OL2", "OL3"]]
    print("mAP@0.5 by occlusion level (%)"); display(piv.round(2))
    piv.round(3).to_csv(OUTPUT_ROOT / "occlusion_map50.csv")

    rows = []
    for model in ol_df["model"].unique():
        if (model, "baseline") not in piv.index or (model, "doam") not in piv.index:
            continue
        b, d = piv.loc[(model, "baseline")], piv.loc[(model, "doam")]
        rows.append({"model": model,
                     "DOAM_OL1_pp": d["OL1"] - b["OL1"], "DOAM_OL2_pp": d["OL2"] - b["OL2"],
                     "DOAM_OL3_pp": d["OL3"] - b["OL3"],
                     "penalty_baseline": b["OL1"] - b["OL3"], "penalty_doam": d["OL1"] - d["OL3"]})
    doam_ol = pd.DataFrame(rows)
    if len(doam_ol):
        print("\nDOAM effect per occlusion level (pp) and occlusion penalty (OL1-OL3):")
        display(doam_ol.round(2))
        levels = ["OL1", "OL2", "OL3"]
        fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
        for _, r in doam_ol.iterrows():
            ax[0].plot(levels, [r["DOAM_OL1_pp"], r["DOAM_OL2_pp"], r["DOAM_OL3_pp"]], marker="o", label=r["model"])
        ax[0].axhline(0, color="gray", ls="--", lw=0.8)
        ax[0].set_title("DOAM effect vs occlusion level"); ax[0].set_xlabel("occlusion level")
        ax[0].set_ylabel("mAP@0.5 gain from DOAM (pp)"); ax[0].legend()
        x = np.arange(len(doam_ol)); w = 0.35
        ax[1].bar(x - w/2, doam_ol["penalty_baseline"], w, label="baseline")
        ax[1].bar(x + w/2, doam_ol["penalty_doam"], w, label="+DOAM")
        ax[1].set_xticks(x); ax[1].set_xticklabels(doam_ol["model"], rotation=15, ha="right")
        ax[1].set_ylabel("occlusion penalty OL1-OL3 (pp, lower = more robust)")
        ax[1].set_title("Occlusion penalty: baseline vs +DOAM"); ax[1].legend()
        plt.tight_layout(); plt.savefig(FIG_DIR / "occlusion_analysis.png", dpi=150, bbox_inches="tight"); plt.show()
else:
    print("Occlusion analysis unavailable (need trained predictions in PRED_STORE).")

### 6.3 Qualitative results and error analysis

The metrics say how often each detector is right; the next two figures show what it gets wrong. Both read the same unified test predictions and apply the same rule as the evaluator: a detection counts if its score is at least 0.3 and it reaches IoU 0.5 with a ground-truth box, matched highest score first.

The first figure takes one model variant and shows examples of each outcome, with the share of test images in that outcome on the left: **correct**, **wrong class** (the box is on the object but the label is wrong), **missed** (no box reaches IoU 0.5), and **extra box** (the object is found but the model also fires somewhere else). Each panel is cropped around the object and labelled with both the true class (green) and the predicted class with its score (red).

The second figure fixes the image and the crop and gives one column per detector variant, so the same object can be compared across models. The panel title reports the predicted class, its score, its IoU with the ground truth, and the resulting verdict.

In [ ]:
# Qualitative error analysis
from matplotlib.patches import Rectangle

GT_COLOR, PRED_COLOR = "#009933", "#e00000"
CASE_ORDER = ["correct", "misclassified", "missed", "false_positive"]
CASE_LABEL = {"correct": "correct\n(right class, IoU>=0.5)",
              "misclassified": "wrong class\n(box found, label wrong)",
              "missed": "missed\n(no box at IoU>=0.5)",
              "false_positive": "extra box\n(object found + false alarm)"}

_TEST_COCO = None
def _test_coco():
    global _TEST_COCO
    if _TEST_COCO is None:
        with contextlib.redirect_stdout(io.StringIO()):
            _TEST_COCO = COCO(str(COCO_ANN["test"]))
    return _TEST_COCO

def _iou_xywh(a, b):
    ax, ay, aw, ah = a; bx, by, bw, bh = b
    ix0, iy0 = max(ax, bx), max(ay, by)
    ix1, iy1 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
    inter = max(0.0, ix1 - ix0) * max(0.0, iy1 - iy0)
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0

def analyse_predictions(model, variant, score_thr=0.3, iou_thr=0.5):
    """Greedy IoU matching of one model's test detections against the ground truth.

    Returns {image_id: {"case", "file_name", "gts", "dets"}}. Each GT keeps the detection
    it was matched to (or None); each detection keeps its IoU with the closest GT. `case`
    describes the whole image and is one of CASE_ORDER.
    """
    preds = json.loads(Path(PRED_STORE[(model, variant)]).read_text())
    coco = _test_coco()
    by_img = defaultdict(list)
    for p in preds:
        if p["score"] >= score_thr:
            by_img[p["image_id"]].append(p)

    out = {}
    for img_id in coco.getImgIds():
        info = coco.loadImgs(img_id)[0]
        gts = [{"bbox": a["bbox"], "cls": a["category_id"], "match": None}
               for a in coco.loadAnns(coco.getAnnIds(imgIds=img_id))]
        dets = [{"bbox": d["bbox"], "cls": d["category_id"], "score": d["score"],
                 "gt_cls": None, "iou": 0.0}
                for d in sorted(by_img.get(img_id, []), key=lambda d: -d["score"])]
        used = set()
        for d in dets:                                   # highest score first
            ious = [_iou_xywh(g["bbox"], d["bbox"]) for g in gts]
            d["iou"] = max(ious) if ious else 0.0
            best, best_iou = None, iou_thr
            for gi, v in enumerate(ious):
                if gi not in used and v >= best_iou:
                    best, best_iou = gi, v
            if best is not None:
                used.add(best); gts[best]["match"] = d; d["gt_cls"] = gts[best]["cls"]

        if any(g["match"] is not None and g["match"]["cls"] != g["cls"] for g in gts):
            case = "misclassified"
        elif any(g["match"] is None for g in gts):
            case = "missed"
        elif any(d["gt_cls"] is None for d in dets):
            case = "false_positive"
        else:
            case = "correct"
        out[img_id] = {"case": case, "file_name": info["file_name"], "gts": gts, "dets": dets}
    return out

def _crop_window(boxes, W, H, pad=1.4, min_side=420):
    """Square window around `boxes`, padded so the object keeps some context."""
    x0 = min(b[0] for b in boxes); y0 = min(b[1] for b in boxes)
    x1 = max(b[0] + b[2] for b in boxes); y1 = max(b[1] + b[3] for b in boxes)
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    half = max(max(x1 - x0, y1 - y0) * (1 + pad) / 2, min_side / 2)
    half = min(half, min(W, H) / 2)
    cx = min(max(cx, half), W - half); cy = min(max(cy, half), H - half)
    return int(cx - half), int(cy - half), int(cx + half), int(cy + half)

def _gt_window(rec, W, H):
    """Window driven by the ground truth only, so different models share one crop."""
    return _crop_window([g["bbox"] for g in rec["gts"]], W, H)

def _draw_case_panel(ax, rec, window=None, max_dets=2, show_labels=True, dets=None):
    """One cropped example: GT boxes green, detections red, both carrying their class name."""
    img = Image.open(COCO_IMG["test"] / rec["file_name"]).convert("RGB")
    dets = rec["dets"][:max_dets] if dets is None else dets[:max_dets]
    if window is None:
        window = _crop_window([g["bbox"] for g in rec["gts"]] + [d["bbox"] for d in dets],
                              img.width, img.height)
    x0, y0, x1, y1 = window
    ax.imshow(np.asarray(img.crop((x0, y0, x1, y1))))
    ax.set_xticks([]); ax.set_yticks([])

    def _box(b, colour, text, y_text, va):
        ax.add_patch(Rectangle((b[0] - x0, b[1] - y0), b[2], b[3],
                               fill=False, edgecolor=colour, linewidth=2))
        if show_labels:
            ax.text(b[0] - x0, y_text, text, color="white", fontsize=7.5, va=va,
                    bbox=dict(facecolor=colour, edgecolor="none", pad=1.2))

    for i, g in enumerate(rec["gts"]):
        _box(g["bbox"], GT_COLOR, f"true {CLASS_NAMES[g['cls']]}",
             g["bbox"][1] - y0 - 4 - i * 20, "bottom")
    for i, d in enumerate(dets):
        _box(d["bbox"], PRED_COLOR, f"pred {CLASS_NAMES[d['cls']]} {d['score']:.2f}",
             d["bbox"][1] + d["bbox"][3] - y0 + 4 + i * 20, "top")
    ax.set_xlim(0, x1 - x0); ax.set_ylim(y1 - y0, 0)
    return window

def _relevant_dets(rec):
    """Boxes worth drawing next to a ground truth: the matched one, or the closest near miss."""
    out = []
    for g in rec["gts"]:
        if g["match"] is not None:
            out.append(g["match"])
        else:
            near = max(rec["dets"], key=lambda d: d["iou"], default=None)
            if near is not None and near["iou"] > 0.1:
                out.append(near)
    return out

def _object_outcome(rec):
    """Per-ground-truth verdict for one image: what was predicted on that object and why."""
    lines, ok = [], True
    for g in rec["gts"]:
        d = g["match"]
        if d is None:
            near = max(rec["dets"], key=lambda z: z["iou"], default=None)
            if near is not None and near["iou"] > 0.1:
                lines.append(f"pred {CLASS_NAMES[near['cls']]} {near['score']:.2f}, IoU {near['iou']:.2f}"
                             f"\n-> missed (IoU < 0.5)")
            else:
                lines.append(f"pred: no box on this object\n-> missed")
            ok = False
        elif d["cls"] != g["cls"]:
            lines.append(f"pred {CLASS_NAMES[d['cls']]} {d['score']:.2f}, IoU {d['iou']:.2f}"
                         f"\n-> wrong class")
            ok = False
        else:
            lines.append(f"pred {CLASS_NAMES[d['cls']]} {d['score']:.2f}, IoU {d['iou']:.2f}"
                         f"\n-> correct")
    n_fp = sum(1 for d in rec["dets"] if d["gt_cls"] is None)
    if n_fp:
        lines.append(f"+{n_fp} false alarm" + ("s" if n_fp > 1 else ""))
    return "\n".join(lines), ok

def _summarise(rec, max_dets=2):
    """Two/three line caption: the true class, the predicted class(es), and why it failed."""
    true = ", ".join(CLASS_NAMES[g["cls"]] for g in rec["gts"]) or "-"
    shown, extra = rec["dets"][:max_dets], max(0, len(rec["dets"]) - max_dets)
    pred = ", ".join(f"{CLASS_NAMES[d['cls']]} {d['score']:.2f}" for d in shown) or "nothing above threshold"
    if extra:
        pred += f" (+{extra} more)"
    text = f"true: {true}\npred: {pred}"
    if rec["case"] == "missed":
        best = max((d["iou"] for d in rec["dets"]), default=0.0)
        text += f"\nbest box IoU {best:.2f} < 0.5"
    return text

def show_qualitative(model="RT-DETR", variant="baseline", per_case=3, score_thr=0.3):
    """One row per outcome, `per_case` examples each, for a single model variant."""
    if (model, variant) not in PRED_STORE:
        print("No predictions for", (model, variant)); return
    analysis = analyse_predictions(model, variant, score_thr=score_thr)
    rng = random.Random(SEED)
    picks = {c: rng.sample(sorted(k for k, v in analysis.items() if v["case"] == c),
                           min(per_case, sum(v["case"] == c for v in analysis.values())))
             for c in CASE_ORDER}
    rows = [c for c in CASE_ORDER if picks[c]]

    fig, axes = plt.subplots(len(rows), per_case, figsize=(4.2 * per_case, 4.6 * len(rows)),
                             squeeze=False)
    for r, case in enumerate(rows):
        n_case = sum(v["case"] == case for v in analysis.values())
        for c in range(per_case):
            ax = axes[r][c]
            if c >= len(picks[case]):
                ax.axis("off"); continue
            rec = analysis[picks[case][c]]
            _draw_case_panel(ax, rec)
            ax.set_title(_summarise(rec), fontsize=9)
            if c == 0:
                ax.set_ylabel(f"{CASE_LABEL[case]}\n{n_case} images, {100*n_case/len(analysis):.1f}%",
                              fontsize=9.5, fontweight="bold")
    fig.suptitle(f"{model} / {variant} - test images by outcome (score >= {score_thr}, IoU >= 0.5)"
                 f"\ngreen = ground truth, red = prediction", fontsize=12)
    plt.tight_layout(h_pad=2.0, rect=(0, 0, 1, 0.98))
    plt.savefig(FIG_DIR / f"error_analysis_{model.replace(' ', '_').replace('-', '')}_{variant}.png",
                dpi=130, bbox_inches="tight")
    plt.show()

if PRED_STORE:
    tgt = ("RT-DETR", "baseline") if ("RT-DETR", "baseline") in PRED_STORE else next(iter(PRED_STORE))
    show_qualitative(*tgt)

In [ ]:
# Same images

MODEL_ORDER = {"Faster R-CNN": 0, "YOLO26": 1, "RT-DETR": 2}

def compare_models_on_images(variants=("baseline", "doam"), n_images=4, score_thr=0.3,
                             prefer_disagreement=True, image_ids=None):
    keys = sorted((k for k in PRED_STORE if k[1] in variants),
                  key=lambda k: (MODEL_ORDER.get(k[0], 9), list(variants).index(k[1])))
    if not keys:
        print("No predictions for variants", variants); return
    analyses = {k: analyse_predictions(*k, score_thr=score_thr) for k in keys}
    coco = _test_coco()

    if image_ids is None:
        pool = sorted(coco.getImgIds())
        if prefer_disagreement:      # images the detectors do not agree on are the informative ones
            mixed = [i for i in pool if len({analyses[k][i]["case"] == "correct" for k in keys}) > 1]
            pool = mixed or pool
        image_ids = random.Random(SEED).sample(pool, min(n_images, len(pool)))

    fig, axes = plt.subplots(len(image_ids), len(keys),
                             figsize=(3.6 * len(keys), 4.2 * len(image_ids)), squeeze=False)
    for r, img_id in enumerate(image_ids):
        info = coco.loadImgs(img_id)[0]
        window = _gt_window(analyses[keys[0]][img_id], info["width"], info["height"])
        for c, key in enumerate(keys):
            rec, ax = analyses[key][img_id], axes[r][c]
            _draw_case_panel(ax, rec, window=window, max_dets=2, show_labels=False,
                             dets=_relevant_dets(rec))
            true_cls = ", ".join(CLASS_NAMES[g["cls"]] for g in rec["gts"])
            verdict, ok = _object_outcome(rec)
            ax.set_title(f"{key[0]} / {key[1]}\n{verdict}", fontsize=9,
                         color=GT_COLOR if ok else PRED_COLOR)
            if c == 0:
                ax.set_ylabel(f"true: {true_cls}\n{rec['file_name']}", fontsize=9,
                              fontweight="bold", color=GT_COLOR)
    fig.suptitle("Same test images, every detector - green box = true object, "
                 "red box = the detection this model put on it"
                 "\ntitle colour: green = right class at IoU >= 0.5, red = wrong class, box too loose, "
                 "or nothing found", fontsize=12)
    plt.tight_layout(h_pad=2.6, rect=(0, 0, 1, 0.97))
    plt.savefig(FIG_DIR / "error_analysis_model_comparison.png", dpi=130, bbox_inches="tight")
    plt.show()

if PRED_STORE:
    compare_models_on_images()

### 6.4 DOAM learned attention

For one test image, each row shows a model's trained +DOAM front module: the input, its learned attention map, and the enhanced input. The colour bar auto-fits each map's 2nd to 98th percentile. The gamma and std in each title show how much that DOAM engaged: a near-zero std is an almost uniform map (little spatial re-weighting).

In [ ]:
def _load_doam_front(source):
    front = DOAM().to(DEVICE)
    if source == "Faster R-CNN":
        p = CKPT_DIR / "faster_rcnn_doam.pth"
        if not p.exists(): return None
        sd = torch.load(p, map_location=DEVICE)
        front.load_state_dict({k[6:]: v for k, v in sd.items() if k.startswith("front.")})
    else:
        register_doam_in_ultralytics()
        run = "yolo26_doam" if source == "YOLO26" else "rtdetr_doam"
        p = OUTPUT_ROOT / "ultra_runs" / run / "weights" / "best.pt"
        if not p.exists(): return None
        model = torch.load(p, map_location=DEVICE, weights_only=False)["model"]
        front.load_state_dict(model.model[0].float().state_dict())   # DOAM is layer 0 of the custom YAML
    front.eval()
    return front

def show_doam_attention(split="test", img_id=None, sources=("Faster R-CNN", "YOLO26", "RT-DETR"),
                        vmin=None, vmax=None):
    # One row per model on a shared image: input | learned attention | enhanced.
    # Colour scale auto-fits each map's 2-98 percentile unless vmin/vmax are given.
    coco = COCO(str(COCO_ANN[split]))
    if img_id is None:
        img_id = random.Random(SEED).choice(coco.getImgIds())
    info = coco.loadImgs(img_id)[0]
    img = Image.open(COCO_IMG[split] / info["file_name"]).convert("RGB").resize((IMGSZ, IMGSZ))
    t = torch.from_numpy(np.asarray(img).transpose(2, 0, 1)).float().div(255).unsqueeze(0).to(DEVICE)
    fig, ax = plt.subplots(len(sources), 3, figsize=(12, 4 * len(sources))); ax = np.atleast_2d(ax)
    for r, source in enumerate(sources):
        front = _load_doam_front(source)
        if front is None:
            for k in range(3): ax[r, k].axis("off")
            ax[r, 0].set_title(f"{source}: no checkpoint"); continue
        with torch.no_grad():
            attn = front.attention_map(t)[0, 0].cpu().numpy()
            enhanced = front(t)[0].clamp(0, 1).cpu().numpy().transpose(1, 2, 0)
        lo = vmin if vmin is not None else float(np.percentile(attn, 2))
        hi = vmax if vmax is not None else float(np.percentile(attn, 98))
        if hi <= lo: hi = lo + 1e-3
        ax[r, 0].imshow(img); ax[r, 0].axis("off"); ax[r, 0].set_title("input" if r == 0 else "")
        m = ax[r, 1].imshow(attn, cmap="jet", vmin=lo, vmax=hi); ax[r, 1].axis("off")
        fig.colorbar(m, ax=ax[r, 1], fraction=0.046)
        ax[r, 1].set_title(f"{source}  gamma={float(front.gamma):+.3f}  std={attn.std():.3f}")
        ax[r, 2].imshow(enhanced); ax[r, 2].axis("off"); ax[r, 2].set_title("enhanced" if r == 0 else "")
    plt.tight_layout(); plt.savefig(FIG_DIR / "doam_attention_trained.png", dpi=130, bbox_inches="tight"); plt.show()

show_doam_attention()

---
## Section 7: Discussion

**Detector comparison (RQ1).** RT-DETR is the most accurate detector (mAP@0.5 89.55), but YOLO26n reaches almost the same mAP@0.5:0.95 (0.388 against 0.390) at a fraction of the cost (2.38M parameters and about 101 FPS, versus 31.99M and 63 FPS), so it is the better accuracy-to-cost choice. Faster R-CNN scores 85.11 at IoU 0.5 yet has the lowest mAP@0.5:0.95 and the lowest recall of the three, which means its boxes are loose and it misses more objects than the headline number suggests.

**DOAM (RQ2, RQ3, RQ4).** DOAM is not a general accuracy booster on these modern detectors: overall it is +0.63 pp on YOLO26, -1.46 pp on RT-DETR, and -0.53 pp on Faster R-CNN. The occlusion breakdown explains why. On YOLO26 the gain is concentrated at the hardest level (OL3 +1.99 pp) with the easy level unchanged, and the occlusion penalty (OL1 mAP minus OL3 mAP) falls from 3.60 to 1.68, which is the de-occlusion behaviour the original paper describes. On RT-DETR the penalty also falls, from 5.14 to 2.69, but only because OL1 drops by 2.28 pp while OL3 stays flat, so it reflects damage to easy images rather than help on hard ones. On Faster R-CNN, already almost occlusion flat, DOAM only adds noise. DOAM therefore transfers to the CNN one-stage detector but not to the Transformer or the two-stage detector, and since it adds about 10% latency on YOLO and 37% on RT-DETR, it is only worth its cost on YOLO26 under heavy occlusion. For reference, plain stronger augmentation with no DOAM (the model_specific runs) raises YOLO26 by +1.84 pp, more than DOAM itself.

**Strengths and weaknesses.** The comparison is fair on the axis that matters: baseline versus +DOAM is controlled inside each stack (one split, one image size, one pycocotools evaluator on exported predictions), and the occlusion analysis separates a real effect on YOLO from a misleading penalty drop on RT-DETR that the overall mAP hides. The weakness is that DOAM's benefit is small and specific to one detector, and that all three detectors are weakest on Straight_Knife, the minority and most occluded class, so class imbalance among visually similar cutters is unresolved.

**Limitations and future work.** Cross-detector comparison is not fully controlled, because the torchvision and Ultralytics stacks differ in optimiser, augmentation, and batch size, so the RQ1 gaps reflect the training recipe as well as the architecture; the within-detector DOAM comparison does not have this issue. Results use a single seed: we made the DOAM training path deterministic, since the original RIA used non-deterministic pooling and upsampling that made single YOLO runs swing between 82.2 and 85.5, so each seed is now reproducible but seed variance is not yet measured. The clearest next steps are to run two or three seeds per DOAM variant to confirm YOLO's OL3 gain, to add SSD for a direct comparison with the original paper, and to address the Straight_Knife weakness with class-balanced training.

## Section 8: Conclusion
Under a unified split and evaluator, RT-DETR is the most accurate detector and YOLO26n is the best accuracy-to-cost trade-off, while the Faster R-CNN baseline is neither. DOAM does not improve these detectors in general; its value is narrow. It acts as an occlusion specialist on YOLO26, adding +1.99 pp at the highest occlusion level and roughly halving the occlusion penalty, which reproduces the mechanism of the original paper, but it does not transfer to the Transformer detector and is redundant on Faster R-CNN. DOAM is therefore worth its cost only for the CNN one-stage detector under heavy occlusion.

## References
1. Wei et al. Occluded Prohibited Items Detection: An X-ray Security Inspection Benchmark and De-occlusion Attention Module (DOAM). ACM MM 2020.
2. Ren et al. Faster R-CNN. NeurIPS 2015. (torchvision `fasterrcnn_resnet50_fpn`, version TODO.)
3. Ultralytics YOLO26, `yolo26n`, version TODO.
4. Lv et al. RT-DETR: DETRs Beat YOLOs on Real-time Object Detection. CVPR 2024. (Ultralytics `rtdetr-l`, version TODO.)
5. Lin et al. Microsoft COCO, with `pycocotools` evaluation.